In [2]:
import pandas as pd 
import numpy as np 
import os
from pathlib import Path

In [ ]:
# load datasets 

dataDIR = Path().resolve().parent / "csvs/raw"
outputDIR = Path().resolve().parent / "csvs/processed"

goalsDF = pd.read_csv(dataDIR / "goals.csv")
cautionsDF = pd.read_csv(dataDIR / "cautions.csv")
subsDF = pd.read_csv(dataDIR / "subs.csv")

In [11]:
goalsDF.head(5)

,game,player,team,min,added_time,md
0,LUGAZI-vs-CALVARY,Simon Peter Odeke,CALVARY,11,No,1.0
1,LUGAZI-vs-CALVARY,Paul Wasswa,LUGAZI,19,No,1.0
2,LUGAZI-vs-CALVARY,Sharif Saaka,LUGAZI,29,No,1.0
3,LUGAZI-vs-CALVARY,Ashiraf Mulindi,LUGAZI,79,No,1.0
4,KITARA-vs-KCCA,Emmanuel Alex Wasswa,KITARA,4,NaN,1.0


In [23]:
goalsDF = goalsDF[goalsDF.isna().sum(axis=1) < 3]

In [20]:
goalsDF['min'].unique()

array(['11', '19', '29', '79', '4', '83', '14', '13', '26', '76', '37',
       '24', '63', '10', '49', '44', '68', '30', '60', '88', '35', '39',
       '25', '53', '28', '82', '3', '85', '90(+3)', '17', '32', '71',
       '16', '23', '90(+1)', '22', '70', '18', '42', '27', '45', '54',
       '77', '45(+3)', '51', '87', '2', '65', '43', '47', '62', '12',
       '46', '72', '78', '89', '36', '64', '41', '75', '45(+1)', '66',
       '1', '8', '57', '90', '74', '67', '34', '58', '90(+2)', '50', '31',
       '55', '59', '38', '33', '45(+2)', '56', '81', '20', '69', '52',
       '48', '15', '86', '90(+4)', '61', '5'], dtype=object)

In [24]:
goalsDF.info()

<class 'pandas.core.frame.DataFrame'>
Index: 197 entries, 0 to 196
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   game        197 non-null    object 
 1   player      197 non-null    object 
 2   team        197 non-null    object 
 3   min         197 non-null    object 
 4   added_time  14 non-null     object 
 5   md          197 non-null    float64
dtypes: float64(1), object(5)
memory usage: 10.8+ KB


In [25]:
goalsDF.tail(5)


,game,player,team,min,added_time,md
192,LUGAZI-vs-VIPERS,Kiza Arafat Usama,VIPERS,8,NaN,11.0
193,LUGAZI-vs-VIPERS,Junior Yunus Sentamu,VIPERS,74,NaN,11.0
194,LUGAZI-vs-VIPERS,Ashiraf Mulindi,LUGAZI,89,NaN,11.0
195,VILLA-vs-CALVARY,Aslam Ssemakula,Villa,5,NaN,11.0
196,ENTEBBE-vs-KITARA,John Kokas Alou,ENTEBBE,34,NaN,11.0


In [29]:
goalsDF['added_time'] = goalsDF['min'].astype(str).str.contains('(', regex=False).map({True: 'yes', False: 'no'})

def _parse_min(v):
    s = str(v).replace('(', '').replace(')', '')
    if '+' in s:
        left, right = s.split('+')
        return int(left) + int(right)
    return int(s)
goalsDF['minute'] = goalsDF['min'].apply(_parse_min).astype(int)

conditions = [
    goalsDF['minute'] <= 45,
    (goalsDF['minute'] > 45) & (goalsDF['minute'] < 60) & (goalsDF['added_time'] == 'yes'),
    (goalsDF['minute'] > 45) & (goalsDF['minute'] <= 90) & (goalsDF['added_time'] == 'no'),
    goalsDF['minute'] > 90
]
choices = [1, 1, 2, 2]
goalsDF['period'] = np.select(conditions, choices)

In [30]:
goalsDF.info()

<class 'pandas.core.frame.DataFrame'>
Index: 197 entries, 0 to 196
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   game        197 non-null    object 
 1   player      197 non-null    object 
 2   team        197 non-null    object 
 3   min         197 non-null    object 
 4   added_time  197 non-null    object 
 5   md          197 non-null    float64
 6   minute      197 non-null    int64  
 7   period      197 non-null    int64  
dtypes: float64(1), int64(2), object(5)
memory usage: 13.9+ KB


In [31]:
goalsDF.head(10)

,game,player,team,min,added_time,md,minute,period
0,LUGAZI-vs-CALVARY,Simon Peter Odeke,CALVARY,11,no,1.0,11,1
1,LUGAZI-vs-CALVARY,Paul Wasswa,LUGAZI,19,no,1.0,19,1
2,LUGAZI-vs-CALVARY,Sharif Saaka,LUGAZI,29,no,1.0,29,1
3,LUGAZI-vs-CALVARY,Ashiraf Mulindi,LUGAZI,79,no,1.0,79,2
4,KITARA-vs-KCCA,Emmanuel Alex Wasswa,KITARA,4,no,1.0,4,1
5,KITARA-vs-KCCA,Sharifu Ssengendo,KCCA,83,no,1.0,83,2
6,EXPRESS-vs-UPDF,Habert Asiimwe,EXPRESS,14,no,1.0,14,1
7,ENTEBBE-vs-BUHIMBA,John Wesley kisaakye,BUHIMBA,13,no,1.0,13,1
8,POLICE-vs-MBARARA,Abudshakur Ramsey Jemba,POLICE,26,no,1.0,26,1
9,POLICE-vs-MBARARA,Reagan Matege,MBARARA,76,no,1.0,76,2
